# Grounded probability gain — CH · T4×2
Hai notebook dùng CÙNG bộ chuẩn bị, cùng khởi tạo,2.000ảnh,500update/nhánh.
C và H luôn khởi tạo lại từ A + nhánh zero; H KHÔNG tiếp tục checkpoint C.

**Lần đầu:** chạy C/H ở MODE='prepare' với CPU để chép cache có sẵn.
Lưu toàn thư mục `grounded_gain_preparation` thành dataset private; gắn dataset
đó vào cả hai tài khoản. Sau đó đổi C/H sang MODE='train' và chạy hai notebook
C/H và T song song bằng T4×2, Internet ON, Save & Run All.

Bước prepare không trích đặc trưng mới hoặc huấn luyện. Không cần UPT mới,
ảnh gốc hay API token. Các path đầu vào nằm ở cell tiếp theo.
Chưa có model run/GPU preflight trên máy local; các checks torch chạy ở đây.
Grounding trên held-out chưa có labelbank: ghi UNAVAILABLE, không coi heatmap
hay loss train là xác nhận ngữ nghĩa/factuality. Chỉ end checkpoint; không sweep/test/rerank.

In [ ]:
from pathlib import Path
MODE='prepare'
ARMS=['C', 'H']
PREPARED_ROOT=None  # Path thư mục chứa receipt.json; None tự tìm đúng metadata hash.
AVAILABLE_DEVICE_GPU_HOURS=25.0  # User xác nhận25giờ MỖI tài khoản; dùng làm ngân sách bảo thủ, không cộng hai tài khoản.
QUOTA_UNIT='device_gpu_hours'
REPORTED_QUOTA=dict(hours_per_account=25,source='user2026-10-27',display_unit='Kaggle GPU hours; device budget entered conservatively',maximum_session='not supplied')
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
PREPARED_OUTPUT=Path('/kaggle/working/grounded_gain_preparation')
OUTPUT_ROOT=Path('/kaggle/working')
assert MODE in ('prepare','train')
assert ARMS==['C', 'H'],'Preserve matched C/H or T protocol.'
if MODE=='train':
    assert AVAILABLE_DEVICE_GPU_HOURS is not None and QUOTA_UNIT=='device_gpu_hours','Record verified per-account GPU balance/unit first.'
    assert AVAILABLE_DEVICE_GPU_HOURS >= 4.4,'Insufficient conservative allocation for this account; do not combine accounts.'
print('Grounded gain',MODE,ARMS,'— bootstrap started',flush=True)


In [ ]:
import hashlib,json,os,shutil,signal,subprocess,sys,tempfile,threading,time,traceback,zipfile
os.environ.update(PYTHONUNBUFFERED='1',HDF5_USE_FILE_LOCKING='FALSE',PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
REPO=Path(tempfile.mkdtemp(prefix='grounded-gain-ch-'))/'repo'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
LOG=OUTPUT_ROOT/'grounded_gain_ch_notebook.log'
PROJECT_COMMIT='28d24f4e6f7157cf0959139271b31c207df75de7'
SOURCE_HASHES={'experiments/semantic_grounding_method/gain_data.py': '46bdc7d379ba9f008f020bf5aa2c7fafeda3809301336deede2de7ae5e03a112', 'experiments/semantic_grounding_method/gain_model.py': 'db5e4c1eda61ab89ca4f5400cb44163f08535a8dbb4fcc4485e782f6380c3283', 'experiments/semantic_grounding_method/gain_runtime.py': '329f5e60c49686a06f093ac685481078ddb5ebf203be41ab54e4837f6aa61781', 'experiments/semantic_grounding_method/test_gain_data.py': 'd3125cad7aba61370c0963ea4f4bc5d1a2baef8724a6400677b9686855494ead', 'experiments/semantic_grounding_method/test_gain_model.py': '70f73ec7c9bc19f49bf6a26d02f4c5241a6391597c66c08fdcd28880a95191d2', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/tokenizer.py': 'df2ace0025aa7d4e12c289ab7bca68046b48d4b6c2b5c12606a19ff5a50a638e', 'captioning/config.py': '4766d1bf82bf82b5199616c7d90ca74242f011e26f08db21b154b1c7d58e72bc', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1'}
METADATA_SHA='62f43f771aec9feba31a2fbbb6fe7db2878f259979faac378b74a754822c7b67'
def fail(message):
    (OUTPUT_ROOT/'grounded_gain_ch_failure.json').write_text(json.dumps(dict(error=str(message),stack=traceback.format_stack())))
    raise RuntimeError(message)
def run(args,cwd=None,timeout=900):
    args=list(map(str,args));print('\nRunning:',' '.join(args),flush=True)
    with LOG.open('a',encoding='utf-8') as log:
        log.write('\nRunning: '+' '.join(args)+'\n');log.flush()
        try:
            process=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                text=True,errors='replace',bufsize=1,start_new_session=True)
        except Exception as error:
            log.write(traceback.format_exc());log.flush();fail(str(error))
        def drain():
            for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        reader=threading.Thread(target=drain,daemon=True);reader.start();start=time.monotonic()
        try:
            while process.poll() is None:
                if time.monotonic()-start>timeout:
                    os.killpg(process.pid,signal.SIGTERM)
                    try:process.wait(timeout=10)
                    except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
                    fail('INCOMPLETE_COMPUTATIONAL_NO_GO: fixed timeout exceeded')
                reader.join(timeout=20)
                if process.poll() is None:print('Worker(s) active; elapsed',round(time.monotonic()-start,1),'s; see stage log.',flush=True)
            reader.join()
            if process.returncode:fail(f'Command failed ({process.returncode}); preserve {LOG}')
        finally:
            if process.poll() is None:os.killpg(process.pid,signal.SIGTERM);process.wait()
            reader.join(timeout=5)
def preserve_review(output):
    # stdlib fallback must not run source guards or hide the original failure.
    try:
        output=Path(output);output.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(LOG,output/'notebook.log')
        with zipfile.ZipFile(str(output)+'_review.zip','w',zipfile.ZIP_DEFLATED) as archive:
            for p in sorted(output.rglob('*')):
                if p.is_file() and p.suffix in ('.json','.jsonl','.log','.py','.md'):archive.write(p,p.relative_to(output))
        print('SEND REVIEW:',str(output)+'_review.zip',flush=True)
    except Exception as error:
        print('Could not build review ZIP; preserve original output/log:',output,LOG,repr(error),flush=True)
REPO.mkdir()
run(['git','init',REPO]);run(['git','remote','add','origin','https://github.com/Supzxjee/Image_Captioning.git'],cwd=REPO)
run(['git','fetch','--depth','1','origin',PROJECT_COMMIT],cwd=REPO)
run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO)
if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()!=PROJECT_COMMIT:fail('Wrong project commit')
os.environ['PYTHONPATH']=str(REPO)
HERE=REPO/'experiments/semantic_grounding_method'
METADATA=HERE/'data/gain_metadata.json.gz'
if hashlib.sha256(METADATA.read_bytes()).hexdigest()!=METADATA_SHA:fail('Metadata digest mismatch')
for name,expected in SOURCE_HASHES.items():
    if hashlib.sha256((REPO/name).read_bytes()).hexdigest()!=expected:fail('Source digest mismatch: '+name)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2','h5py','numpy','Pillow','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
CHAIR=REPO.parent/'chair'
run(['git','init',CHAIR]);run(['git','remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],cwd=CHAIR)
run(['git','fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],cwd=CHAIR)
run(['git','checkout','--detach','FETCH_HEAD'],cwd=CHAIR)
assert hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()=='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'
run([sys.executable,'-u',HERE/'test_gain_data.py'],cwd=REPO)
CONFIG_BASE=dict(checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),metadata=str(METADATA),
    chair=str(CHAIR),sources=SOURCE_HASHES,project_commit=PROJECT_COMMIT)
RUNTIME=HERE/'gain_runtime.py'


In [ ]:
if MODE=='prepare':
    assert not PREPARED_OUTPUT.exists(),'Preserve previous output; change path or use a fresh session.'
    cfg=dict(CONFIG_BASE,legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),detections=str(DETECTIONS),
        prepared_output=str(PREPARED_OUTPUT),output=str(PREPARED_OUTPUT))
    config=REPO.parent/'prepare.json';config.write_text(json.dumps(cfg,indent=2))
    try:
        run([sys.executable,'-u',RUNTIME,'prepare','--config',config],cwd=REPO,timeout=1800)
    except Exception as error:
        PREPARED_OUTPUT.mkdir(parents=True,exist_ok=True)
        (PREPARED_OUTPUT/'failure.json').write_text(json.dumps(dict(status='INCOMPLETE_COMPUTATIONAL_NO_GO',error=str(error),traceback=traceback.format_exc())))
        raise
    finally:
        if PREPARED_OUTPUT.is_dir():
            preserve_review(PREPARED_OUTPUT)
    print('PREPARATION COMPLETE. Save entire folder as a private dataset, attach to both accounts:',PREPARED_OUTPUT,flush=True)
    print('Then switch C/H MODE to train; T already uses train. This pass has NOT trained C/H/T.',flush=True)


In [ ]:
if MODE=='train':
    import torch
    assert torch.cuda.device_count()==2,'Choose T4x2 on EACH notebook.'
    assert all('T4' in torch.cuda.get_device_name(i) for i in (0,1)),'Protocol requires the same T4x2 hardware on both accounts.'
    if PREPARED_ROOT is None:
        matches=[]
        for p in Path('/kaggle/input').rglob('receipt.json'):
            try:r=json.loads(p.read_text())
            except (ValueError,OSError):continue
            if r.get('metadata_sha256')==METADATA_SHA and r.get('status')=='PREPARATION_COMPLETE':matches.append(p.parent)
        assert len(matches)==1,f'Attach exactly one complete preparation dataset; found {len(matches)}. Set PREPARED_ROOT explicitly if needed.'
        PREPARED_ROOT=matches[0]
    sys.path.insert(0,str(HERE))
    from gain_data import checked_cache
    checked_cache(PREPARED_ROOT,METADATA_SHA,hash_payloads=False)  # rank0 verifies full payloads before each GPU phase.
    run([sys.executable,'-u',HERE/'test_gain_model.py'],cwd=REPO,timeout=120)
    for arm in ARMS:
        output=OUTPUT_ROOT/('grounded_gain_'+arm)
        if (output/'summary.json').is_file():
            previous=json.loads((output/'summary.json').read_text())
            expected_receipt=hashlib.sha256((Path(PREPARED_ROOT)/'receipt.json').read_bytes()).hexdigest()
            assert previous['status']=='COMPLETE_WAIT_MATCHED_CHT' and previous['arm']==arm and previous['updates']==500
            assert previous['sources']==SOURCE_HASHES and previous['preparation_receipt_sha256']==expected_receipt
            assert hashlib.sha256((output/'checkpoint.bin').read_bytes()).hexdigest()==previous['checkpoint_sha256']
            print('Preserve and reuse completed matched arm:',arm,flush=True)
            continue
        assert not output.exists(),'Preserve previous arm; use a fresh session/output root.'
        output.mkdir(parents=True)
        cfg=dict(CONFIG_BASE,prepared=str(PREPARED_ROOT),output=str(output),arm=arm,
            quota=dict(available=AVAILABLE_DEVICE_GPU_HOURS,unit=QUOTA_UNIT,reported=REPORTED_QUOTA))
        config=output/'protocol.json';config.write_text(json.dumps(cfg,indent=2))
        for n in SOURCE_HASHES:
            target=output/'source'/n;target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(REPO/n,target)
        try:
            run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',RUNTIME,'preflight','--config',config],cwd=REPO,timeout=600)
            cfg['arm_started_unix']=time.time();config.write_text(json.dumps(cfg,indent=2))
            run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',RUNTIME,'worker','--config',config],cwd=REPO,timeout=2700)
            remaining=2700-(time.time()-cfg['arm_started_unix'])
            assert remaining>0,'Arm time cap exhausted before CPU endpoint scoring.'
            run([sys.executable,'-u',RUNTIME,'evaluate','--config',config],cwd=REPO,timeout=remaining)
        except Exception as error:
            (output/'failure.json').write_text(json.dumps(dict(status='INCOMPLETE_COMPUTATIONAL_NO_GO',error=str(error),arm=arm,traceback=traceback.format_exc())))
            raise
        finally:
            preserve_review(output)
        print('DONE',arm,'SEND:',str(output)+'_review.zip','KEEP:',output/'checkpoint.bin',flush=True)


Gửi đủ3ZIP review C/H/T; giữ riêng checkpoint.bin và nguyên dataset chuẩn bị.
Một nhánh hoàn tất chưa đủ kết luận. So sánh pairedCI10.000samples với cùngA/dev1000,
kiểmT−A>=.01,CI(T−A,T−C,T−H)>0,CHAIRkhôngxấu,recall>=A−.005.
No-Go dừng recipe, không chứng minh trần kiến trúc. GPU/time/precisionfailure là
INCOMPLETE, không phải kết quả chất lượng âm. Không sửa tolerance/loss/seed để cứu lượt chạy.
Không có held-out phrase-grounding gold trong gói này: không tuyên bố causalgrounding
hoặc đóng góp paper chỉ từ loss/heatmap đẹp. Các giới hạn ghi trong protocol/review.